# STAC Catalog Builder — Freshem Chloride (Zarr)

This notebook generates a STAC catalog for the preprocessed Freshem **chloride**
Zarr datacube. It follows the same publish workflow as
`scripts/12_crop_production.ipynb` (configure -> build collection/items -> save
locally -> upload to cloud) but the datacube has spatial `x`/`y` dimensions plus
an additional (non-uniform, layer-based) `z` dimension instead of `time`/`station`.

**Prerequisite:** run `notebooks/16_chloride.ipynb` first to
produce the Zarr store and `metadata_chloride.json`.

There is no GeoServer/WMS step and no Mapbox tiling — the frontend reads the
Zarr datacube directly (same approach as the crop-production stations).

**Cloud upload (steps 8-9) is optional and guarded by `RUN_UPLOAD`.** Leave it
`False` until credentials/bucket targets are confirmed for Freshem.

In [1]:
import datetime
import json
import os
from pathlib import Path
from posixpath import join as urljoin

import pystac
import xarray as xr
from pystac import Summaries
from pystac.stac_io import DefaultStacIO

from coclicodata.coclico_stac.datacube import add_datacube
from coclicodata.coclico_stac.extension import CoclicoExtension, CollectionCoclicoExtension
from coclicodata.coclico_stac.layouts import CoCliCoZarrLayout
from coclicodata.coclico_stac.templates import gen_zarr_asset, get_template_collection
from coclicodata.etl.extract import zero_terminated_bytes_as_str


## 1) Configure paths and options

In [2]:
import os

stac_data_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))
repo_root = stac_data_repo.parent.parent.parent  # .../global-coastal-atlas
stac_dir = stac_data_repo / "current"

freshem_repo = Path(os.environ.get("FRESHEM_REPO", r"C:\Ocean\Work\Projects\2026\Freshem\Tools\freshem_stac"))
data_dir = freshem_repo / "processed" / "stac_folder" / "chloride"
metadata_path = data_dir / "metadata_chloride.json"
google_cred_path = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\google_credentials.json"
)

ZARR_STORE = {
    "id": "chloride",
    "filename": "chloride.zarr",
    "title": "Chloride",
    "description": "Freshem chloride datacube (x, y, z)",
}

RUN_UPLOAD = True  # set True once GCS bucket/credentials for Freshem are confirmed
gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"
google_cred_path = google_cred_path

# Datacube / frontend settings: regular x/y grid, no time dimension, non-uniform z
X_DIMENSION = "x"
Y_DIMENSION = "y"
TEMPORAL_DIMENSION = False  # must be False (not None), add_datacube() indexes ds[dim] otherwise
ADDITIONAL_DIMENSIONS = ["z"]
DIMENSIONS_TO_IGNORE = []
PLOT_X_AXIS = "z"
PLOT_TYPE = "line"
PLOT_SERIES = ""

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

with open(metadata_path) as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "SPATIAL_EXTENT",
    "COLLECTION_ID",
    "TITLE",
    "DESCRIPTION",
    "LICENSE",
    "PROVIDERS",
    "KEYWORDS",
    "UNITS",
    "MEDIA_TYPE",
    "CRS",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(f"Missing required metadata keys in {metadata_path}: {', '.join(missing_keys)}")

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
href_prefix = urljoin(gcs_protocol, bucket_name or "<bucket>", bucket_proj, proj_name)

zarr_path = data_dir / ZARR_STORE["filename"]
if not zarr_path.exists():
    raise FileNotFoundError(f"Zarr store not found: {zarr_path}")

print("Zarr input:", zarr_path)
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Native CRS:", metadata["CRS"])
print("Spatial extent (EPSG:4326):", metadata["SPATIAL_EXTENT"])


Zarr input: C:\Ocean\Work\Projects\2026\Freshem\Tools\freshem_stac\processed\stac_folder\chloride\chloride.zarr
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\freshem_chloride
Collection id: freshem_chloride
Native CRS: EPSG:28992
Spatial extent (EPSG:4326): [3.64028, 51.375148, 3.960967, 51.616396]


## 2) Define STAC helper functions

In [3]:
def parse_license(metadata: dict) -> str:
    license_ = metadata["LICENSE"]
    if "Creative Commons" in license_ and "4.0" in license_:
        return "CC-BY-4.0"
    return license_


def create_providers(metadata: dict) -> list[pystac.Provider]:
    return [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[pystac.provider.ProviderRole.PRODUCER, pystac.provider.ProviderRole.LICENSOR],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[pystac.provider.ProviderRole.PROCESSOR, pystac.provider.ProviderRole.HOST],
            url="https://deltares.nl",
        ),
    ]


def create_extent(metadata: dict) -> pystac.Extent:
    """Freshem datasets have no temporal dimension: use an open [None, None] interval."""
    return pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[None, None]]),
    )


def bbox_geometry_from_extent(spatial_extent: list[float]) -> tuple[dict, list[float]]:
    west, south, east, north = spatial_extent
    bbox = [west, south, east, north]
    geometry = {
        "type": "Polygon",
        "coordinates": [[[west, south], [east, south], [east, north], [west, north], [west, south]]],
    }
    return geometry, bbox


def create_collection(metadata: dict, collection_id: str, *, template_fp: Path) -> pystac.Collection:
    extent = create_extent(metadata)
    collection = get_template_collection(
        template_fp=str(template_fp),
        collection_id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        keywords=metadata["KEYWORDS"],
        license=parse_license(metadata),
        spatial_extent=[metadata["SPATIAL_EXTENT"]],
        temporal_extent=extent.temporal.intervals,
        providers=create_providers(metadata),
    )

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": metadata.get("MEDIA_TYPE", "application/vnd+zarr"),
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["DESCRIPTION"],
            "xarray:storage_options": {"token": "google_default"},
        }
    }
    return collection


def open_zarr_store(zarr_path: Path) -> xr.Dataset:
    ds = xr.open_zarr(zarr_path, consolidated=False)
    return zero_terminated_bytes_as_str(ds)


def zarr_storage_href(store: dict, href_prefix: str) -> str:
    return urljoin(href_prefix, store["filename"])


def process_zarr_store(store: dict, zarr_path: Path, metadata: dict, href_prefix: str) -> tuple[pystac.Item, xr.Dataset]:
    ds = open_zarr_store(zarr_path)
    geometry, bbox = bbox_geometry_from_extent(metadata["SPATIAL_EXTENT"])

    item = pystac.Item(
        id=store["id"],
        geometry=geometry,
        bbox=bbox,
        datetime=datetime.datetime.now(datetime.timezone.utc),
        properties={"store": store["id"], "description": store["description"]},
    )
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)
    item.properties["deltares:item_key"] = store["id"]

    storage_href = zarr_storage_href(store, href_prefix)
    item.add_asset("data", gen_zarr_asset(store["title"], storage_href))
    return item, ds


def add_datacube_from_store(collection: pystac.Collection, ds: xr.Dataset, *, reference_system: str) -> pystac.Collection:
    return add_datacube(
        stac_obj=collection,
        ds=ds,
        temporal_dimension=TEMPORAL_DIMENSION,
        x_dimension=X_DIMENSION,
        y_dimension=Y_DIMENSION,
        additional_dimensions=ADDITIONAL_DIMENSIONS,
        reference_system=reference_system,
    )


def build_variable_summaries(ds: xr.Dataset, label: str) -> dict:
    options = []
    for var_name in ds.data_vars:
        long_name = ds[var_name].attrs.get("long_name", str(var_name))
        options.append({"label": long_name, "value": var_name})
    return {"label": label, "options": options}


def apply_coclico_collection_props(collection: pystac.Collection, metadata: dict) -> None:
    """Set Coclico frontend props directly (see 12_crop_production.ipynb for rationale)."""
    schema_uri = CoclicoExtension.get_schema_uri()
    if collection.stac_extensions is None:
        collection.stac_extensions = [schema_uri]
    elif not any(uri.endswith("json-schema/schema.json") for uri in collection.stac_extensions):
        collection.stac_extensions.append(schema_uri)

    coclico_ext = CollectionCoclicoExtension(collection)
    coclico_ext.units = metadata["UNITS"]
    coclico_ext.plot_series = PLOT_SERIES
    coclico_ext.plot_x_axis = PLOT_X_AXIS
    coclico_ext.plot_type = PLOT_TYPE
    coclico_ext.min_ = 0
    coclico_ext.linear_gradient = []


def verify_local_stac_outputs(stac_dir: Path, collection_id: str, *, expected_items: int) -> None:
    collection_json = stac_dir / collection_id / "collection.json"
    item_jsons = [p for p in (stac_dir / collection_id).rglob("*.json") if p.name != "collection.json"]
    catalog_has_child = False
    catalog_path = stac_dir / "catalog.json"
    if catalog_path.is_file():
        with open(catalog_path, encoding="utf-8") as f:
            catalog_has_child = f"./{collection_id}/collection.json" in json.dumps(json.load(f))

    missing = []
    if not collection_json.is_file():
        missing.append(str(collection_json))
    if len(item_jsons) < expected_items:
        missing.append(f"{expected_items} item JSON files (found {len(item_jsons)})")
    if not catalog_has_child:
        missing.append(f"catalog child link ./{collection_id}/collection.json")

    if missing:
        raise FileNotFoundError(
            "STAC outputs incomplete after save. Re-run steps 3-6 in the same kernel, "
            "then step 7 again.\nMissing: " + "; ".join(missing)
        )

    print(f"Verified: {collection_json}")
    print(f"Verified: {len(item_jsons)} item JSON file(s)")
    print(f"Verified: catalog.json links to ./{collection_id}/collection.json")


def restore_original_var_names(ds: xr.Dataset, safe_to_original: dict | None = None) -> xr.Dataset:
    """Rename Zarr-safe vars back to original Freshem names (from attrs or map)."""
    if safe_to_original is None:
        raw = ds.attrs.get("FRESHEM_VAR_NAME_MAP")
        if not raw:
            return ds
        safe_to_original = json.loads(raw)
    rename = {
        k: v
        for k, v in safe_to_original.items()
        if k in ds.data_vars and v not in ds.dims and v not in ds.coords and v not in ds.data_vars
    }
    skipped = {
        k: v for k, v in safe_to_original.items() if k in ds.data_vars and k not in rename
    }
    if skipped:
        print(
            "Not restoring original name(s) because they collide with an existing "
            f"dimension/variable: {skipped}. Keeping the Zarr-safe name(s)."
        )
    return ds.rename(rename) if rename else ds


## 3) Create the STAC collection

In [4]:
template_fp = stac_dir / "template" / "collection.json"
collection = create_collection(metadata, collection_id, template_fp=template_fp)
layout = CoCliCoZarrLayout()

print(f"Collection ready: {collection.id}")
print(f"Title: {metadata['TITLE']}")
print(f"Provider: {metadata['PROVIDERS']['name']}")
print(f"Media type: {metadata['MEDIA_TYPE']}")
print(f"Coclico extensions: {collection.stac_extensions}")


Collection ready: freshem_chloride
Title: Freshem groundwater chloride concentration model (Zeeland)
Provider: Deltares
Media type: application/vnd+zarr
Coclico extensions: ['https://raw.githubusercontent.com/openearth/coclicodata/feat/update-deltares-stac-properties/json-schema/schema.json', 'https://stac-extensions.github.io/item-assets/v1.0.0/schema.json']


## 4) Add datacube metadata

In [5]:
ds = open_zarr_store(zarr_path)

collection = add_datacube_from_store(collection, ds, reference_system=metadata["CRS"])

collection.add_asset(ZARR_STORE["id"], gen_zarr_asset(ZARR_STORE["title"], zarr_storage_href(ZARR_STORE, href_prefix)))

collection.summaries = Summaries({})
collection.summaries.add(ZARR_STORE["id"], build_variable_summaries(ds, ZARR_STORE["title"]))

apply_coclico_collection_props(collection, metadata)

print("cube:dimensions:", list(collection.extra_fields.get("cube:dimensions", {}).keys()))
print("cube:variables:", len(collection.extra_fields.get("cube:variables", {})))
print("collection assets:", list(collection.assets.keys()))
print("summaries:", list(collection.summaries.to_dict().keys()))


cube:dimensions: ['x', 'y', 'z']
cube:variables: 28
collection assets: ['chloride']
summaries: ['chloride']


## 5) Build the STAC item

One item per Zarr store (a single datacube here).

In [6]:
item, _ = process_zarr_store(ZARR_STORE, zarr_path, metadata, href_prefix)
item_href = stac_dir / collection_id / ZARR_STORE["id"] / f"{ZARR_STORE['id']}.json"
item.set_self_href(str(item_href))
collection.add_item(item, strategy=layout)

print(f"Created item: {item.id}")
print(f"Data asset href: {item.assets['data'].href}")


Created item: chloride
Data asset href: https://storage.googleapis.com/gca-data-public/gca/freshem_chloride/chloride.zarr


## 6) Save STAC catalog locally

In [7]:
stac_io = DefaultStacIO()

(stac_dir / collection_id / ZARR_STORE["id"]).mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED, dest_href=str(stac_dir), stac_io=stac_io)

collection.validate_all()
catalog.validate_all()
verify_local_stac_outputs(stac_dir, collection_id, expected_items=1)
print(f"STAC saved to: {stac_dir}")


Removed existing child: freshem_chloride
Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\freshem_chloride\collection.json
Verified: 1 item JSON file(s)
Verified: catalog.json links to ./freshem_chloride/collection.json
STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 7) Upload Zarr store to Google Cloud (optional)

Skip until `RUN_UPLOAD = True` and bucket/credentials are confirmed for Freshem.

In [8]:
if RUN_UPLOAD:
    import gcsfs

    from coclicodata.etl.cloud_utils import load_google_credentials

    load_google_credentials(google_token_fp=google_cred_path)
    fs = gcsfs.GCSFileSystem(gcs_project, token=os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

    target = urljoin(bucket_name, bucket_proj, proj_name, ZARR_STORE["filename"])
    print(f"Uploading {ZARR_STORE['filename']} -> {target} ...")
    if fs.exists(target):
        print(f"  Removing existing remote store: {target}")
        fs.rm(target, recursive=True)
    fs.put(str(zarr_path), target, recursive=True)
    print(f"Done: {ZARR_STORE['id']}")
else:
    print("RUN_UPLOAD is False, skipping Zarr upload.")


C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:339: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:352: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Google Application Credentials load into environment.
Uploading chloride.zarr -> gca-data-public/gca/freshem_chloride/chloride.zarr ...
  Removing existing remote store: gca-data-public/gca/freshem_chloride/chloride.zarr
Done: chloride


## 8) Upload STAC catalog to Google Cloud (optional)

In [9]:
if RUN_UPLOAD:
    from coclicodata.etl.cloud_utils import dir_to_google_cloud, file_to_google_cloud, load_google_credentials

    verify_local_stac_outputs(stac_dir, collection_id, expected_items=1)
    load_google_credentials(google_token_fp=google_cred_path)

    dir_to_google_cloud(
        dir_path=str(stac_dir / collection_id),
        gcs_project=gcs_project,
        bucket_name=bucket_name,
        bucket_proj=bucket_proj,
        dir_name=urljoin(stac_cloud_name, collection_id),
    )

    file_to_google_cloud(
        file_path=str(stac_dir / "catalog.json"),
        gcs_project=gcs_project,
        bucket_name=bucket_name,
        bucket_proj=bucket_proj,
        dir_name=stac_cloud_name,
        file_name="catalog.json",
    )
    print(f"Uploaded STAC collection: gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/{collection_id}/")
    print(f"Uploaded catalog: gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/catalog.json")
else:
    print("RUN_UPLOAD is False, skipping STAC catalog upload.")


Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\freshem_chloride\collection.json
Verified: 1 item JSON file(s)
Verified: catalog.json links to ./freshem_chloride/collection.json
Google Application Credentials load into environment.
Writing to directory at gca-data-public/gca/gca-stac-7/freshem_chloride...
Source directory: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\freshem_chloride
Using put() with recursive=True...
Done!
Uploaded STAC collection: gs://gca-data-public/gca/gca-stac-7/freshem_chloride/
Uploaded catalog: gs://gca-data-public/gca/gca-stac-7/catalog.json
